# 🚀 Retrain Model UMKM Bogor — v3

Notebook ini melatih ulang model **Random Forest** untuk prediksi daya tarik produk UMKM Bogor  
menggunakan dataset hasil preprocessing terbaru (`data_processed/dataset_preprocessed.csv`, 1027 produk).

## Perubahan utama dari v2 → v3
| Aspek | v2 (lama) | v3 (baru) |
|---|---|---|
| Sumber data | `dataset_umkm_bogor.csv` (597 baris) | `data_processed/dataset_preprocessed.csv` (1027 baris) |
| Fitur numerik | rasio_harga, zscore_harga, log_harga, segmen_harga, rating | + `jumlah_log`, `revenue_proxy_log`, `popularity_score` |
| Market stats | Per kategori saja | Per kategori **dan** per sub_kategori |
| Output model | `model_umkm_bogor_v2.joblib` | `model_umkm_bogor_v3.joblib` |
| Market stats file | `market_stats_per_kategori.csv` | `market_stats_v3.csv` |
| Fitur Paling Digemari | ❌ | ✅ popularity stats tersimpan |


## 📦 0. Import Library

In [1]:
import pandas as pd
import numpy as np
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, accuracy_score,
    roc_auc_score, f1_score, precision_score, recall_score
)

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    SASTRAWI_AVAILABLE = True
    stemmer = StemmerFactory().create_stemmer()
    print('✅ Sastrawi tersedia')
except ImportError:
    SASTRAWI_AVAILABLE = False
    stemmer = None
    print('⚠️  Sastrawi tidak ditemukan — text sudah di-clean di preprocessing')

print(f'\n✅ Library siap | pandas {pd.__version__} | sklearn {__import__("sklearn").__version__}')


✅ Sastrawi tersedia

✅ Library siap | pandas 2.2.3 | sklearn 1.6.1


---
## 📁 1. Load Dataset Preprocessed

In [2]:
print('=' * 65)
print('  RETRAIN MODEL UMKM BOGOR v3 — DATASET TERBARU (1027 PRODUK)')
print('=' * 65)

df = pd.read_csv('../data/processed/dataset_preprocessed.csv')
print(f'\n✅ Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'   Kolom: {list(df.columns)}')
print(f'\n   Distribusi kategori:')
print(df['kategori'].value_counts().to_string())
print(f'\n   Distribusi marketplace:')
print(df['marketplace'].value_counts().to_string())
df.head(3)


  RETRAIN MODEL UMKM BOGOR v3 — DATASET TERBARU (1027 PRODUK)

✅ Dataset dimuat: 1027 baris, 36 kolom
   Kolom: ['url_produk', 'nama_produk', 'nama_produk_clean', 'kategori', 'sub_kategori', 'marketplace', 'lokasi', 'lokasi_clean', 'nama_toko', 'harga_produk', 'jumlah_terjual', 'rating', 'harga_log_scaled', 'jumlah_log_scaled', 'rating_scaled', 'revenue_proxy_log', 'popularity_score', 'kategori_encoded', 'sub_kategori_encoded', 'marketplace_encoded', 'lokasi_encoded', 'mp_lazada', 'mp_shopee', 'mp_tokopedia', 'lokasi_Bekasi', 'lokasi_Bogor', 'lokasi_Depok', 'lokasi_Jakarta', 'lokasi_Lainnya', 'lokasi_Tangerang', 'is_bogor', 'is_makanan', 'is_minuman', 'is_fashion', 'is_souvenir', 'harga_tier']

   Distribusi kategori:
kategori
Makanan                 416
Pakaian & Fashion       415
Minuman                 171
Aksesoris & Souvenir     25

   Distribusi marketplace:
marketplace
shopee       438
lazada       342
tokopedia    247


,url_produk,nama_produk,nama_produk_clean,kategori,sub_kategori,marketplace,lokasi,lokasi_clean,nama_toko,harga_produk,...,lokasi_Depok,lokasi_Jakarta,lokasi_Lainnya,lokasi_Tangerang,is_bogor,is_makanan,is_minuman,is_fashion,is_souvenir,harga_tier
0,https://www.tokopedia.com/100persenbogorpisan/...,Gantungan Kunci Bogor / Oleh-oleh Khas Bogor/D...,gantungan kunci dua sisi premium,Aksesoris & Souvenir,Aksesoris & Souvenir,tokopedia,Bogor,Bogor,100persen.bogorpisan,6900.0,...,False,False,False,False,1,0,0,0,1,0
1,https://www.tokopedia.com/21import/keripik-tal...,Keripik Talas Bogor Gurih | Keripik Stik Talas...,keripik talas gurih keripik stik talas asli gr,Makanan,Camilan & Snack,tokopedia,Jakarta Barat,Jakarta,21 import store,51000.0,...,False,True,False,False,0,1,0,0,0,1
2,https://www.tokopedia.com/abdul-rokhim1/kaos-b...,KAOS BOGOR TUGU KUJANG - KAOS BUITENZORG - KAO...,kaos tugu kujang kaos buitenzorg kaos rain cit...,Pakaian & Fashion,Atasan & Pakaian Kasual,tokopedia,Jakarta Barat,Jakarta,ABDUL ROKHIM1,69999.0,...,False,True,False,False,0,0,0,1,0,1


---
## 📊 2. Hitung Market Statistics

In [3]:
print('[Step 2] Menghitung market statistics...')

# ── Statistik per kategori (untuk konteks harga) ──────────────────────────────
stat_per_kategori = df.groupby('kategori')['harga_produk'].agg(
    median_harga_kategori='median',
    mean_harga_kategori='mean',
    std_harga_kategori='std',
    q25_harga_kategori=lambda x: x.quantile(0.25),
    q75_harga_kategori=lambda x: x.quantile(0.75),
).reset_index()

# ── Statistik popularitas per kategori (untuk fitur 'paling digemari') ────────
pop_per_kategori = df.groupby('kategori').agg(
    total_popularity=('popularity_score', 'sum'),
    avg_popularity=('popularity_score', 'mean'),
    total_terjual=('jumlah_terjual', 'sum'),
    avg_rating=('rating', 'mean'),
    jumlah_produk=('nama_produk', 'count'),
).reset_index()

# ── Statistik popularitas per sub_kategori ────────────────────────────────────
pop_per_sub = df.groupby(['kategori', 'sub_kategori']).agg(
    total_popularity=('popularity_score', 'sum'),
    avg_popularity=('popularity_score', 'mean'),
    total_terjual=('jumlah_terjual', 'sum'),
    avg_rating=('rating', 'mean'),
    jumlah_produk=('nama_produk', 'count'),
).reset_index()

# ── Gabungkan dan simpan ───────────────────────────────────────────────────────
market_stats = stat_per_kategori.merge(pop_per_kategori, on='kategori', how='left')
market_stats.to_csv('../data/market_stats_v3.csv', index=False)
pop_per_sub.to_csv('../data/market_stats_sub_kategori_v3.csv', index=False)

print(f'   ✅ market_stats_v3.csv disimpan ({len(market_stats)} kategori)')
print(f'   ✅ market_stats_sub_kategori_v3.csv disimpan ({len(pop_per_sub)} sub_kategori)')

print('\n   Ranking popularitas per kategori:')
print(pop_per_kategori.sort_values('total_popularity', ascending=False).to_string(index=False))

print('\n   Ranking popularitas per sub_kategori (top 10):')
print(pop_per_sub.sort_values('total_popularity', ascending=False).head(10).to_string(index=False))


[Step 2] Menghitung market statistics...
   ✅ market_stats_v3.csv disimpan (4 kategori)
   ✅ market_stats_sub_kategori_v3.csv disimpan (10 sub_kategori)

   Ranking popularitas per kategori:
            kategori  total_popularity  avg_popularity  total_terjual  avg_rating  jumlah_produk
             Makanan       9437.966352       22.687419        96236.5    4.816346            416
   Pakaian & Fashion       5167.074350       12.450782         9207.5    4.948193            415
             Minuman       4062.426397       23.756880        42628.5    4.889474            171
Aksesoris & Souvenir        563.972066       22.558883         4436.5    4.884000             25

   Ranking popularitas per sub_kategori (top 10):
            kategori            sub_kategori  total_popularity  avg_popularity  total_terjual  avg_rating  jumlah_produk
   Pakaian & Fashion Atasan & Pakaian Kasual       4466.576374       12.511418         7673.0    4.945098            357
             Makanan           

---
## ⚙️ 3. Feature Engineering (Fitur Relatif Pasar)

In [4]:
print('[Step 3] Feature Engineering...')

# Merge statistik pasar ke df
df = df.merge(stat_per_kategori, on='kategori', how='left')

# ── Rasio harga vs median kategori ────────────────────────────────────────────
df['rasio_harga'] = (
    df['harga_produk'] / df['median_harga_kategori'].replace(0, np.nan)
).fillna(1.0).clip(upper=50)

# ── Z-score harga dalam kategori ─────────────────────────────────────────────
df['zscore_harga'] = (
    (df['harga_produk'] - df['mean_harga_kategori']) /
    df['std_harga_kategori'].replace(0, 1)
).clip(-5, 5)

# ── Log harga ─────────────────────────────────────────────────────────────────
df['log_harga'] = np.log1p(df['harga_produk'])

# ── Segmen harga (0=murah, 1=menengah, 2=premium) per kategori ───────────────
def segment_harga(row):
    if row['harga_produk'] <= row['q25_harga_kategori']: return 0
    elif row['harga_produk'] <= row['q75_harga_kategori']: return 1
    else: return 2

df['segmen_harga'] = df.apply(segment_harga, axis=1)

# ── Jumlah log (sudah ada tapi recompute untuk safety) ───────────────────────
df['jumlah_log'] = np.log1p(df['jumlah_terjual'])

# ── Revenue proxy log ─────────────────────────────────────────────────────────
# (sudah ada di CSV sebagai revenue_proxy_log, tapi kita recompute)
df['revenue_proxy_log'] = np.log1p(df['harga_produk'] * df['jumlah_terjual'])

# ── Popularity score ──────────────────────────────────────────────────────────
# (sudah ada di CSV, recompute untuk consistency)
df['popularity_score'] = df['rating'] * np.log1p(df['jumlah_terjual'])

print('   ✅ Fitur baru berhasil dibuat:')
fitur_baru = ['rasio_harga', 'zscore_harga', 'log_harga', 'segmen_harga',
              'jumlah_log', 'revenue_proxy_log', 'popularity_score']
print(df[fitur_baru].describe().round(3).to_string())


[Step 3] Feature Engineering...


   ✅ Fitur baru berhasil dibuat:
       rasio_harga  zscore_harga  log_harga  segmen_harga  jumlah_log  revenue_proxy_log  popularity_score
count     1027.000      1027.000   1027.000      1027.000    1027.000           1027.000          1027.000
mean         1.151        -0.000     10.653         0.987       3.844             14.420            18.726
std          0.641         0.999      0.712         0.715       1.681              1.684             8.124
min          0.020        -1.831      6.217         0.000       0.693              8.071             1.386
25%          0.714        -0.709     10.340         0.000       2.485             13.266            12.425
50%          1.000        -0.237     10.758         1.000       3.555             14.036            17.170
75%          1.447         0.534     11.051         1.000       5.416             15.849            26.539
max          4.443         2.809     11.618         2.000       6.300             17.916            31.499


---
## 🏷️ 4. Auto-Labeling

In [5]:
print('[Step 4] Membuat label...')

# Strategi: label = 1 jika jumlah_terjual > median
# Menghasilkan distribusi yang hampir seimbang
median_terjual = df['jumlah_terjual'].median()
df['label'] = (df['jumlah_terjual'] > median_terjual).astype(int)

print(f'   Median jumlah terjual : {median_terjual}')
print(f'   Distribusi label      :')
vc = df['label'].value_counts()
print(f'     Label 0 (Kurang Menarik) : {vc.get(0, 0)} ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'     Label 1 (Menarik)        : {vc.get(1, 0)} ({vc.get(1,0)/len(df)*100:.1f}%)')
print(f'     Rasio balance            : {vc.min()/vc.max():.3f} (1.0 = sempurna)')


[Step 4] Membuat label...
   Median jumlah terjual : 34.0
   Distribusi label      :
     Label 0 (Kurang Menarik) : 514 (50.0%)
     Label 1 (Menarik)        : 513 (50.0%)
     Rasio balance            : 0.998 (1.0 = sempurna)


---
## 🔢 5. Feature Matrix

In [6]:
print('[Step 5] Menyiapkan feature matrix...')

text_feature = 'nama_produk_clean'
cat_features  = ['kategori', 'sub_kategori']
num_features  = [
    'rasio_harga',       # harga relatif median kategori
    'zscore_harga',      # z-score harga dalam kategori
    'log_harga',         # log1p(harga) — distribusi stabil
    'segmen_harga',      # 0=murah / 1=menengah / 2=premium
    'rating',            # rating produk
    'jumlah_log',        # log1p(jumlah_terjual) — NEW v3
    'revenue_proxy_log', # log1p(harga × jumlah) — NEW v3
    'popularity_score',  # rating × log1p(jumlah) — NEW v3
]

all_features = [text_feature] + cat_features + num_features

# Filter hanya kolom yang ada
available = [c for c in all_features if c in df.columns]
missing   = [c for c in all_features if c not in df.columns]
if missing:
    print(f'   ⚠️  Kolom tidak ditemukan: {missing}')

X = df[available]
y = df['label']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'   Total data     : {len(X)}')
print(f'   Train+Val set  : {len(X_train_val)}')
print(f'   Test set       : {len(X_test)}')
print(f'   Fitur teks     : 1 (TF-IDF 1000 features)')
print(f'   Fitur kategori : {len(cat_features)} (OHE)')
print(f'   Fitur numerik  : {len(num_features)}')


[Step 5] Menyiapkan feature matrix...
   Total data     : 1027
   Train+Val set  : 821
   Test set       : 206
   Fitur teks     : 1 (TF-IDF 1000 features)
   Fitur kategori : 2 (OHE)
   Fitur numerik  : 8


---
## 🔧 6. Pipeline Preprocessing

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1000, ngram_range=(1, 2), sublinear_tf=True), text_feature),
        ('cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num',  StandardScaler(), num_features),
    ]
)

print('✅ ColumnTransformer pipeline dibuat')
print(f'   TF-IDF  : max_features=1000, ngram=(1,2), sublinear_tf=True')
print(f'   OHE     : kategori + sub_kategori')
print(f'   Scaler  : StandardScaler → {num_features}')


✅ ColumnTransformer pipeline dibuat
   TF-IDF  : max_features=1000, ngram=(1,2), sublinear_tf=True
   OHE     : kategori + sub_kategori
   Scaler  : StandardScaler → ['rasio_harga', 'zscore_harga', 'log_harga', 'segmen_harga', 'rating', 'jumlah_log', 'revenue_proxy_log', 'popularity_score']


---
## 📈 7. Cross Validation Baseline (5-Fold)

In [8]:
print('[Step 7] Cross Validation baseline (5-Fold Stratified)...')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1))
])

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_results = cross_validate(
    rf_baseline, X_train_val, y_train_val,
    cv=skf, scoring=scoring, return_train_score=False
)

print(f"\n   {'Metrik':<15} {'Mean':>8} {'±Std':>8}")
print('   ' + '-' * 35)
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f'   {metric:<15} {scores.mean():.4f}   ±{scores.std():.4f}')


[Step 7] Cross Validation baseline (5-Fold Stratified)...



   Metrik              Mean     ±Std
   -----------------------------------
   accuracy        0.9903   ±0.0091
   precision       0.9928   ±0.0096
   recall          0.9878   ±0.0189
   f1              0.9902   ±0.0093
   roc_auc         0.9998   ±0.0004


---
## 🔍 8. Hyperparameter Tuning (GridSearchCV)

In [9]:
print('[Step 8] Hyperparameter Tuning (GridSearchCV)...')
print('   Ini mungkin butuh beberapa menit...')

param_grid = {
    'classifier__n_estimators'   : [100, 200, 300],
    'classifier__max_depth'      : [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf' : [1, 2],
    'classifier__class_weight'   : ['balanced', None],
}

rf_for_grid = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=1))
])

grid_search = GridSearchCV(
    rf_for_grid, param_grid, cv=skf,
    scoring='roc_auc', n_jobs=1, verbose=1
)
grid_search.fit(X_train_val, y_train_val)

print(f'\n   ✅ Selesai!')
print(f'   Best Parameters : {grid_search.best_params_}')
print(f'   Best AUC-ROC CV : {grid_search.best_score_:.4f}')


[Step 8] Hyperparameter Tuning (GridSearchCV)...
   Ini mungkin butuh beberapa menit...
Fitting 5 folds for each of 72 candidates, totalling 360 fits



   ✅ Selesai!
   Best Parameters : {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
   Best AUC-ROC CV : 0.9998


---
## 🧪 9. Evaluasi di Holdout Test Set

In [10]:
print('[Step 9] Evaluasi di holdout test set...')

best_rf = grid_search.best_estimator_
y_pred  = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print(f"\n   {'Metrik':<15} {'Score':>8}")
print('   ' + '-' * 25)
print(f"   {'Accuracy':<15} {accuracy_score(y_test, y_pred):.4f}")
print(f"   {'Precision':<15} {precision_score(y_test, y_pred):.4f}")
print(f"   {'Recall':<15} {recall_score(y_test, y_pred):.4f}")
print(f"   {'F1 Score':<15} {f1_score(y_test, y_pred):.4f}")
print(f"   {'AUC-ROC':<15} {roc_auc_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Kurang Menarik', 'Menarik']))


[Step 9] Evaluasi di holdout test set...

   Metrik             Score
   -------------------------
   Accuracy        0.9854
   Precision       0.9902
   Recall          0.9806
   F1 Score        0.9854
   AUC-ROC         0.9996

                precision    recall  f1-score   support

Kurang Menarik       0.98      0.99      0.99       103
       Menarik       0.99      0.98      0.99       103

      accuracy                           0.99       206
     macro avg       0.99      0.99      0.99       206
  weighted avg       0.99      0.99      0.99       206



---
## ⚖️ 10. Perbandingan: Random Forest vs Logistic Regression

In [11]:
print('[Step 10] Perbandingan model (LR vs RF)...')

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000, class_weight='balanced',
        random_state=42, solver='lbfgs'
    ))
])

lr_cv = cross_validate(lr_pipeline, X_train_val, y_train_val, cv=skf, scoring=scoring)
lr_pipeline.fit(X_train_val, y_train_val)
lr_pred  = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

rf_pred  = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:, 1]

comparison = {
    'Metrik': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC'],
    'Random Forest v3': [
        accuracy_score(y_test, rf_pred), precision_score(y_test, rf_pred),
        recall_score(y_test, rf_pred), f1_score(y_test, rf_pred),
        roc_auc_score(y_test, rf_proba),
    ],
    'Logistic Regression': [
        accuracy_score(y_test, lr_pred), precision_score(y_test, lr_pred),
        recall_score(y_test, lr_pred), f1_score(y_test, lr_pred),
        roc_auc_score(y_test, lr_proba),
    ],
}

df_comp = pd.DataFrame(comparison).set_index('Metrik').round(4)
print('\n   === TABEL PERBANDINGAN FINAL ===')
print(df_comp.to_string())

def highlight_winner(row):
    styles = [''] * len(row)
    styles[row.values.argmax()] = 'font-weight: bold; color: green'
    return styles

df_comp.style.apply(highlight_winner, axis=1)


[Step 10] Perbandingan model (LR vs RF)...



   === TABEL PERBANDINGAN FINAL ===
           Random Forest v3  Logistic Regression
Metrik                                          
Accuracy             0.9854               0.9709
Precision            0.9902               1.0000
Recall               0.9806               0.9417
F1 Score             0.9854               0.9700
AUC-ROC              0.9996               0.9997


,Random Forest v3,Logistic Regression
Metrik,,
Accuracy,0.985400,0.970900
Precision,0.990200,1.000000
Recall,0.980600,0.941700
F1 Score,0.985400,0.970000
AUC-ROC,0.999600,0.999700


---
## ✅ 11. Sanity Check Bisnis

In [12]:
print('[Step 11] Sanity check bisnis...')

contoh_df = df[df['label'] == 1].head(3)
for _, row in contoh_df.iterrows():
    kategori   = row['kategori']
    stat       = stat_per_kategori[stat_per_kategori['kategori'] == kategori].iloc[0]
    median_kat = stat['median_harga_kategori']
    q25, q75   = stat['q25_harga_kategori'], stat['q75_harga_kategori']

    def make_test_row(harga, jumlah_est=30):
        rasio  = min(harga / max(median_kat, 1), 50)
        zscore = np.clip((harga - stat['mean_harga_kategori']) / max(stat['std_harga_kategori'], 1), -5, 5)
        seg    = 0 if harga <= q25 else (1 if harga <= q75 else 2)
        return pd.DataFrame([{
            'nama_produk_clean': row['nama_produk_clean'],
            'kategori'         : kategori,
            'sub_kategori'     : row['sub_kategori'],
            'rasio_harga'      : rasio,
            'zscore_harga'     : zscore,
            'log_harga'        : np.log1p(harga),
            'segmen_harga'     : seg,
            'rating'           : row['rating'],
            'jumlah_log'       : np.log1p(jumlah_est),
            'revenue_proxy_log': np.log1p(harga * jumlah_est),
            'popularity_score' : row['rating'] * np.log1p(jumlah_est),
        }])

    prob_normal = best_rf.predict_proba(make_test_row(median_kat))[0][1]
    prob_mahal  = best_rf.predict_proba(make_test_row(median_kat * 20))[0][1]

    print(f'\n   Produk  : {row["nama_produk"][:55]}...')
    print(f'   Kategori: {kategori} | Median Rp{median_kat:,.0f}')
    print(f'   Harga normal (1x median): peluang laku {prob_normal*100:.1f}%')
    print(f'   Harga gila  (20x median): peluang laku {prob_mahal*100:.1f}%')
    print(f'   → Penurunan: {(prob_normal-prob_mahal)*100:.1f} poin', '✅' if prob_normal > prob_mahal else '⚠️')


[Step 11] Sanity check bisnis...

   Produk  : Gantungan Kunci Bogor / Oleh-oleh Khas Bogor/Dua Sisi/ ...
   Kategori: Aksesoris & Souvenir | Median Rp25,000
   Harga normal (1x median): peluang laku 24.0%
   Harga gila  (20x median): peluang laku 33.0%
   → Penurunan: -9.0 poin ⚠️

   Produk  : Kopi Gula Liong Bulan Legend Khas Bogor 1 pak 2 renceng...
   Kategori: Minuman | Median Rp38,500
   Harga normal (1x median): peluang laku 33.0%
   Harga gila  (20x median): peluang laku 44.0%
   → Penurunan: -11.0 poin ⚠️

   Produk  : OLEH-OLEH KHAS BOGOR INDONESIA| PROMO GANTUNGAN KUNCI B...
   Kategori: Aksesoris & Souvenir | Median Rp25,000
   Harga normal (1x median): peluang laku 32.0%
   Harga gila  (20x median): peluang laku 25.0%
   → Penurunan: 7.0 poin ✅


---
## 💾 12. Retrain Seluruh Data → Simpan Model v3

In [13]:
print('[Step 12] Retrain dengan SELURUH data → simpan model final...')

best_params = grid_search.best_params_

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params['classifier__n_estimators'],
        max_depth=best_params['classifier__max_depth'],
        min_samples_split=best_params['classifier__min_samples_split'],
        min_samples_leaf=best_params['classifier__min_samples_leaf'],
        class_weight=best_params['classifier__class_weight'],
        random_state=42, n_jobs=1
    ))
])

final_model.fit(X, y)

joblib.dump(final_model, '../models/model_umkm_bogor_v3.joblib')
print(f'   ✅ Model disimpan  → model_umkm_bogor_v3.joblib')
print(f'   ✅ Market stats    → market_stats_v3.csv')
print(f'   ✅ Sub-kat stats   → market_stats_sub_kategori_v3.csv')

import os
size_mb = os.path.getsize('../models/model_umkm_bogor_v3.joblib') / 1024 / 1024
print(f'   📦 Ukuran model   : {size_mb:.1f} MB')

print('\n' + '=' * 65)
print('  SELESAI! Ringkasan perubahan v2 → v3:')
print('=' * 65)
print(f'''
  Dataset   : dataset_umkm_bogor.csv (597) → dataset_preprocessed.csv ({len(df)})
  Fitur baru: jumlah_log, revenue_proxy_log, popularity_score
  Market stats: per kategori + per sub_kategori (untuk fitur Paling Digemari)
  Model     : model_umkm_bogor_v2.joblib → model_umkm_bogor_v3.joblib
''')


[Step 12] Retrain dengan SELURUH data → simpan model final...


   ✅ Model disimpan  → model_umkm_bogor_v3.joblib
   ✅ Market stats    → market_stats_v3.csv
   ✅ Sub-kat stats   → market_stats_sub_kategori_v3.csv
   📦 Ukuran model   : 1.3 MB

  SELESAI! Ringkasan perubahan v2 → v3:

  Dataset   : dataset_umkm_bogor.csv (597) → dataset_preprocessed.csv (1027)
  Fitur baru: jumlah_log, revenue_proxy_log, popularity_score
  Market stats: per kategori + per sub_kategori (untuk fitur Paling Digemari)
  Model     : model_umkm_bogor_v2.joblib → model_umkm_bogor_v3.joblib

